In [1]:
import json
import random
import time
from collections import Counter
from pathlib import Path

import numpy as np
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers

try:
    from tqdm.auto import tqdm
except ImportError:                       
    def tqdm(x, **kwargs):
        return x

import tokenizers
print("numpy      :", np.__version__)
print("tokenizers :", tokenizers.__version__)

numpy      : 2.4.6
tokenizers : 0.22.2


## Configuration

In [ ]:
PROJECT_ROOT = Path("..")

FINAL_FILE    = PROJECT_ROOT / "data" / "final" / "fitness_final.jsonl"
TOKENIZER_DIR = PROJECT_ROOT / "tokenizer"
DATASET_DIR   = PROJECT_ROOT / "dataset"

TOKENIZER_FILE = TOKENIZER_DIR / "tokenizer.json"
TRAIN_BIN      = DATASET_DIR / "train.bin"
VAL_BIN        = DATASET_DIR / "val.bin"
META_FILE      = DATASET_DIR / "meta.json"

for d in (TOKENIZER_DIR, DATASET_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------- Tokenizer ----------
VOCAB_SIZE     = 8000          
MIN_FREQUENCY  = 2             
SPECIAL_TOKENS = ["<eos>"]

TOKENIZER_SAMPLE_DOCS = 200_000

# ---------- Dataset ----------
BLOCK_SIZE = 256               # context length: model bir qadamda kuradigan token soni
VAL_RATIO  = 0.01              # hujjatlarning 1% i validationga
SEED       = 1337

MAX_DOCS = None

random.seed(SEED)
np.random.seed(SEED)

print("Final corpus :", FINAL_FILE.resolve())
print("Tokenizer    :", TOKENIZER_FILE.resolve())
print("Dataset      :", DATASET_DIR.resolve())

Final corpus : C:\TrainerAI_LLM\our_llm\data\final\fitness_final.jsonl
Tokenizer    : C:\TrainerAI_LLM\our_llm\tokenizer\tokenizer.json
Dataset      : C:\TrainerAI_LLM\our_llm\dataset


In [3]:
FINAL_FILE

WindowsPath('../data/final/fitness_final.jsonl')

## 1. Datani yuklash

In [4]:
def load_jsonl(path, max_docs=None):
    """JSONL fayldan matnlar ro'yxatini o'qiydi."""
    texts = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_docs is not None and i >= max_docs:
                break
            texts.append(json.loads(line)["text"])
    return texts


if not FINAL_FILE.exists():
    raise FileNotFoundError(
        str(FINAL_FILE) + " topilmadi.\n"
        "Avval data/experiment.ipynb ni to'liq ishga tushiring - u shu fileni yasadi."
    )

docs = load_jsonl(FINAL_FILE, max_docs=MAX_DOCS)

total_chars = sum(len(t) for t in docs)
total_words = sum(len(t.split()) for t in docs)

print("Hujjatlar    : {:,}".format(len(docs)))
print("Belgilar     : {:,}".format(total_chars))
print("So'zlar      : {:,}".format(total_words))
print("O'rtacha     : {:.0f} belgi / hujjat".format(total_chars / len(docs)))
print()
print("Namuna:")
print(docs[0][:300])

Hujjatlar    : 170
Belgilar     : 94,501
So'zlar      : 15,045
O'rtacha     : 556 belgi / hujjat

Namuna:
Instruction: What are some practical steps I can take to improve my overall health and well-being?
Output: 1. Develop a consistent exercise routine – Exercise is essential for physical and mental health. Aim for at least 30 minutes of physical activity five days a week.

2. Follow a healthy diet – I


---

# A. TOKENIZER



## 2. Comparison of word-level and character-level usage 

In [5]:
sample = docs[:20_000]                     
sample_chars = sum(len(t) for t in sample)

# --- character-level ---
char_vocab = set()
for t in sample:
    char_vocab.update(t)
char_tokens = sample_chars                

# --- word-level ---
word_counter = Counter()
for t in sample:
    word_counter.update(t.split())
word_tokens = sum(word_counter.values())

header = "{:<14}{:>12}{:>16}{:>16}".format("Method", "Vocabulary", "Tokens", "Token/belgi")
print(header)
print("-" * len(header))
print("{:<14}{:>12,}{:>16,}{:>16.3f}".format(
    "character", len(char_vocab), char_tokens, char_tokens / sample_chars))
print("{:<14}{:>12,}{:>16,}{:>16.3f}".format(
    "word", len(word_counter), word_tokens, word_tokens / sample_chars))
print()

singletons = sum(1 for c in word_counter.values() if c == 1)
print("Faqat 1 marta uchragan wordlar: {:,} / {:,}  ({:.1%})".format(
    singletons, len(word_counter), singletons / len(word_counter)))
print()
print("Eng kam uchraydigan 10 ta word:", [w for w, _ in word_counter.most_common()[-10:]])

Method          Vocabulary          Tokens     Token/belgi
----------------------------------------------------------
character               79          94,501           1.000
word                 3,280          15,045           0.159

Faqat 1 marta uchragan wordlar: 1,884 / 3,280  (57.4%)

Eng kam uchraydigan 10 ta word: ['variety.', 'Analyze', 'trainer', 'perspectives', 'insights.', 'urine', 'color', 'status.', 'electrolytes', 'impairs']


## 3. BPE: Merge Rules

In [6]:
def apply_merge(word, pair):
    """word - bo'laklar tuple'i. pair - birlashtiriladigan juftlik."""
    out, i = [], 0
    while i < len(word):
        if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
            out.append(word[i] + word[i + 1])      # ikkitasini bittaga qo'shdik
            i += 2
        else:
            out.append(word[i])
            i += 1
    return tuple(out)


def toy_bpe_train(text, num_merges, verbose=True):
    """Klassik so'z-darajasidagi BPE. '_' - so'z oxiri belgisi."""
    freqs = Counter(text.split())
    words = {tuple(list(w) + ["_"]): c for w, c in freqs.items()}

    merges = []
    for step in range(num_merges):
        pairs = Counter()
        for word, freq in words.items():
            for pair in zip(word, word[1:]):
                pairs[pair] += freq
        if not pairs:
            break

        best, count = pairs.most_common(1)[0]
        if count < 2:
            break

        words = {apply_merge(w, best): c for w, c in words.items()}
        merges.append(best)

        if verbose:
            print("merge {:2d}: {!r} + {!r}  ->  {!r}   (uchradi: {})".format(
                step + 1, best[0], best[1], best[0] + best[1], count))

    return merges, words


toy_text = ("low low low low low lower lower newest newest newest "
            "newest newest newest widest widest widest")

print("Matn:", toy_text)
print()
toy_merges, toy_words = toy_bpe_train(toy_text, num_merges=10)

Matn: low low low low low lower lower newest newest newest newest newest newest widest widest widest

merge  1: 'e' + 's'  ->  'es'   (uchradi: 9)
merge  2: 'es' + 't'  ->  'est'   (uchradi: 9)
merge  3: 'est' + '_'  ->  'est_'   (uchradi: 9)
merge  4: 'l' + 'o'  ->  'lo'   (uchradi: 7)
merge  5: 'lo' + 'w'  ->  'low'   (uchradi: 7)
merge  6: 'n' + 'e'  ->  'ne'   (uchradi: 6)
merge  7: 'ne' + 'w'  ->  'new'   (uchradi: 6)
merge  8: 'new' + 'est_'  ->  'newest_'   (uchradi: 6)
merge  9: 'low' + '_'  ->  'low_'   (uchradi: 5)
merge 10: 'w' + 'i'  ->  'wi'   (uchradi: 3)


In [7]:
print("Urgatilgandan keyin suzlar qanday parchaga bulingan:\n")
for word, freq in sorted(toy_words.items(), key=lambda kv: -kv[1]):
    print("  {:>3} x  {}".format(freq, list(word)))

print("\nMerge rules:")
for i, (a, b) in enumerate(toy_merges, 1):
    print("  {:2d}. {!r} + {!r}".format(i, a, b))

Urgatilgandan keyin suzlar qanday parchaga bulingan:

    6 x  ['newest_']
    5 x  ['low_']
    3 x  ['wi', 'd', 'est_']
    2 x  ['low', 'e', 'r', '_']

Merge rules:
   1. 'e' + 's'
   2. 'es' + 't'
   3. 'est' + '_'
   4. 'l' + 'o'
   5. 'lo' + 'w'
   6. 'n' + 'e'
   7. 'ne' + 'w'
   8. 'new' + 'est_'
   9. 'low' + '_'
  10. 'w' + 'i'


In [8]:
def toy_bpe_encode(word, merges):
    tokens = tuple(list(word) + ["_"])
    for pair in merges:
        tokens = apply_merge(tokens, pair)
    return list(tokens)


for w in ["lowest", "newer", "wider", "slow", "qwerty"]:
    print("{:>8}  ->  {}".format(w, toy_bpe_encode(w, toy_merges)))

print()
print("'qwerty' uquv matnida umuman yuq edi, lekin <unk> chiqmadi -")
print("u belgilarga bulinib qoldi. Mana shu BPE ning asosiy plus i.")

  lowest  ->  ['low', 'est_']
   newer  ->  ['new', 'e', 'r', '_']
   wider  ->  ['wi', 'd', 'e', 'r', '_']
    slow  ->  ['s', 'low_']
  qwerty  ->  ['q', 'w', 'e', 'r', 't', 'y', '_']

'qwerty' uquv matnida umuman yuq edi, lekin <unk> chiqmadi -
u belgilarga bulinib qoldi. Mana shu BPE ning asosiy plus i.


## 4. Byte-level BPE

In [9]:
s = "Once upon a time"
print("Matn   :", s)
print("Baytlar:", list(s.encode("utf-8")))
print("Bayt soni:", len(s.encode("utf-8")))
print()

byte_alphabet = pre_tokenizers.ByteLevel.alphabet()
print("Byte-level vocabulary hajmi:", len(byte_alphabet)) # hardoim 256! 
print("Namuna:", sorted(byte_alphabet)[:20])

Matn   : Once upon a time
Baytlar: [79, 110, 99, 101, 32, 117, 112, 111, 110, 32, 97, 32, 116, 105, 109, 101]
Bayt soni: 16

Byte-level vocabulary hajmi: 256
Namuna: ['!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4']


## 5. Vocabulary size

In [10]:
D_MODEL = 384

print("d_model = {}, tied weights (embedding = output head)\n".format(D_MODEL))
head = "{:>10}{:>18}{:>16}".format("vocab", "embed params", "ulush (30M da)")
print(head)
print("-" * len(head))
for v in [1_000, 4_000, 8_000, 16_000, 32_000, 50_257, 65_535]:
    p = v * D_MODEL
    mark = "   <-- bizniki" if v == VOCAB_SIZE else ""
    print("{:>10,}{:>18,}{:>15.1%}{}".format(v, p, p / 30_000_000, mark))

print()
print("50257 - GPT-2 ning vocabulary. 65535 - uint16 dagi chegara.")

d_model = 384, tied weights (embedding = output head)

     vocab      embed params  ulush (30M da)
--------------------------------------------
     1,000           384,000           1.3%
     4,000         1,536,000           5.1%
     8,000         3,072,000          10.2%   <-- bizniki
    16,000         6,144,000          20.5%
    32,000        12,288,000          41.0%
    50,257        19,298,688          64.3%
    65,535        25,165,440          83.9%

50257 - GPT-2 ning vocabulary. 65535 - uint16 dagi chegara.


## 6. Special tokens

In [11]:
def tokenizer_corpus_iterator(texts, batch_size=1000):
    for i in range(0, len(texts), batch_size):
        yield from texts[i:i + batch_size]

train_texts_for_tokenizer = docs[:TOKENIZER_SAMPLE_DOCS]
print("Tokenizer o'rgatiladigan hujjatlar: {:,}".format(len(train_texts_for_tokenizer)))

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder       = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),   # 256 baytning hammasi
    min_frequency=MIN_FREQUENCY,
    show_progress=True,
)

t0 = time.time()
tokenizer.train_from_iterator(
    tokenizer_corpus_iterator(train_texts_for_tokenizer),
    trainer=trainer,
    length=len(train_texts_for_tokenizer),
)
print("\nO'rgatish vaqti: {:.1f} soniya".format(time.time() - t0))

tokenizer.save(str(TOKENIZER_FILE))
print("Saqlandi:", TOKENIZER_FILE.resolve())
print("Fayl hajmi: {:.2f} MB".format(TOKENIZER_FILE.stat().st_size / 1024**2))
print("Haqiqiy vocab_size:", tokenizer.get_vocab_size())

Tokenizer o'rgatiladigan hujjatlar: 170

O'rgatish vaqti: 0.0 soniya
Saqlandi: C:\TrainerAI_LLM\our_llm\tokenizer\tokenizer.json
Fayl hajmi: 0.19 MB
Haqiqiy vocab_size: 3106


In [12]:
vocab = tokenizer.get_vocab()       
id_to_token = {i: t for t, i in vocab.items()}

print("Birinchi 20 ta ID (special + baytlar):")
print([id_to_token[i] for i in range(20)])
print()

# Special tokenlar qayerda?
for tok in SPECIAL_TOKENS:
    print("{:>8} -> ID {}".format(tok, tokenizer.token_to_id(tok)))
print()

start = 256 + len(SPECIAL_TOKENS)
print("Birinchi 40 ta O'RGANILGAN token (Ġ = bo'sh joy):")
print([id_to_token[i] for i in range(start, start + 40)])
print()

longest = sorted(vocab.keys(), key=len, reverse=True)[:15]
print("Eng uzun 10 ta token:", longest)

Birinchi 20 ta ID (special + baytlar):
['<eos>', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3']

   <eos> -> ID 0

Birinchi 40 ta O'RGANILGAN token (Ġ = bo'sh joy):
['Ġa', 'in', 're', 'Ġt', 'Ġs', 'ti', 'ou', 'on', 'nd', 'or', 'he', 'Ġand', 'al', 'ing', 'Ġf', 'es', 'at', 'Ġc', 'er', 'Ġw', 'Ġm', 'Ġy', 'en', 'Ġp', 'Ġyou', 'an', 'it', 'Ġb', 'ut', 'Ġto', 'tion', 'le', 'st', 'Ġd', 'Ġre', 'uc', 'Ġyour', 'Ġo', 'ar', 've']

Eng uzun 10 ta token: ['Ġresponsibilities', 'Ġtransportation', 'Ġcardiovascular', 'Ġaccountability', 'Ġopportunities', 'Ġunderstanding', 'Ġencouragement', 'Ġcarbohydrates', 'Ġenvironmental', 'Ġincorporating', 'Ġconcentration', 'Ġmodifications', 'Ġsensitivities', 'Ġsignificantly', 'Ġrelationships']


## 7. Round-trip test

In [13]:
def round_trip(text, verbose=True):
    enc = tokenizer.encode(text)
    back = tokenizer.decode(enc.ids)
    ok = (back == text)
    if verbose:
        status = "OK  " if ok else "XATO"
        print("[{}] {:>3} token  {!r}".format(status, len(enc.ids), text[:60]))
        if not ok:
            print("      kutilgan: {!r}".format(text))
            print("      olingan : {!r}".format(back))
    return ok


edge_cases = [
    "lorem ipsum dolor",
    "hello my name is..",               # kup byte li belgilar (2 byte)
    "emoji: \U0001F600\U0001F680",      # 4 byte li belgilar
    "你好, éàü",                         # xitoycha + aksentli lotin
    "  bir nechta   empty   joy  ",     # empty joylar saqlanadimi?
    "row1\nrow2\n\nrow10",              # yangi qatorlar
    "\ttab bilan boshlandi",
    "46239847 va 3.14159",
    "",                                 # empty satr
    "Hiiiiiiiiiii",
]

print("--- Chekka holatlar ---")
edge_ok = sum(round_trip(t) for t in edge_cases)
print("\n{}/{} o'tdi".format(edge_ok, len(edge_cases)))

assert edge_ok == len(edge_cases), "Chekka holatda round-trip buzildi!"

--- Chekka holatlar ---
[OK  ]   8 token  'lorem ipsum dolor'
[OK  ]  10 token  'hello my name is..'
[OK  ]  14 token  'emoji: 😀🚀'
[OK  ]  14 token  '你好, éàü'
[OK  ]  18 token  '  bir nechta   empty   joy  '
[OK  ]   9 token  'row1\nrow2\n\nrow10'
[OK  ]  12 token  '\ttab bilan boshlandi'
[OK  ]  17 token  '46239847 va 3.14159'
[OK  ]   0 token  ''
[OK  ]  12 token  'Hiiiiiiiiiii'

10/10 o'tdi


In [14]:
tricky = "<eos> matn ichida yozilgan"
enc = tokenizer.encode(tricky)

print("Matn      :", repr(tricky))
print("Tokenlar  :", enc.tokens[:5], "...")
print("IDs       :", enc.ids[:5], "...")
print()
print("decode() default            :", repr(tokenizer.decode(enc.ids)))
print("decode(skip_special=False)  :",
      repr(tokenizer.decode(enc.ids, skip_special_tokens=False)))

Matn      : '<eos> matn ichida yozilgan'
Tokenlar  : ['<eos>', 'Ġm', 'at', 'n', 'Ġ'] ...
IDs       : [0, 277, 273, 78, 221] ...

decode() default            : ' matn ichida yozilgan'
decode(skip_special=False)  : '<eos> matn ichida yozilgan'


In [15]:
held_out = docs[TOKENIZER_SAMPLE_DOCS:] or docs[-1000:]
check_sample = random.sample(held_out, min(2000, len(held_out)))

failures = [t for t in check_sample if tokenizer.decode(tokenizer.encode(t).ids) != t]

print("Tekshirilgan hujjatlar : {:,}".format(len(check_sample)))
print("Xatolar                : {}".format(len(failures)))

assert not failures, "Round-trip buzildi! Tokenizer sozlamalarini tekshiring."
print("\nROUND-TRIP TEST O'TDI - encode/decode lossless.")

Tekshirilgan hujjatlar : 170
Xatolar                : 0

ROUND-TRIP TEST O'TDI - encode/decode lossless.


## 8. Tokenizationni o'lchash: fertility va compression

In [16]:
def measure(tok, texts):
    n_tokens = sum(len(e.ids) for e in tok.encode_batch(texts))
    n_words  = sum(len(t.split()) for t in texts)
    n_chars  = sum(len(t) for t in texts)
    n_bytes  = sum(len(t.encode("utf-8")) for t in texts)
    return {
        "tokens": n_tokens,
        "fertility": n_tokens / n_words,
        "compression": n_chars / n_tokens,
        "bytes_per_token": n_bytes / n_tokens,
    }


eval_sample = random.sample(held_out, min(5000, len(held_out)))   # HELD-OUT
ours = measure(tokenizer, eval_sample)

print("Baholash namunasi: {:,} hujjat (o'rgatishda qatnashmagan)\n".format(len(eval_sample)))
print("Bizning tokenizer (vocab={:,})".format(tokenizer.get_vocab_size()))
print("  fertility       : {:.3f}  token/so'z     (past = yaxshi)".format(ours["fertility"]))
print("  compression     : {:.3f}  belgi/token    (yuqori = yaxshi)".format(ours["compression"]))
print("  bytes per token : {:.3f}".format(ours["bytes_per_token"]))

Baholash namunasi: 170 hujjat (o'rgatishda qatnashmagan)

Bizning tokenizer (vocab=3,106)
  fertility       : 1.412  token/so'z     (past = yaxshi)
  compression     : 4.448  belgi/token    (yuqori = yaxshi)
  bytes per token : 4.459


In [17]:
# GPT-2 bilan solishtirish - AYNAN SHU matnda.
# Internet kerak; bo'lmasa bu katak sakraladi.
try:
    gpt2 = Tokenizer.from_pretrained("gpt2")
    theirs = measure(gpt2, eval_sample)

    head = "{:<22}{:>12}{:>14}{:>14}".format("Tokenizer", "vocab", "fertility", "compression")
    print(head)
    print("-" * len(head))
    print("{:<22}{:>12,}{:>14.3f}{:>14.3f}".format(
        "bizniki", tokenizer.get_vocab_size(), ours["fertility"], ours["compression"]))
    print("{:<22}{:>12,}{:>14.3f}{:>14.3f}".format(
        "GPT-2", gpt2.get_vocab_size(), theirs["fertility"], theirs["compression"]))
    print()
    rel = (ours["tokens"] - theirs["tokens"]) / theirs["tokens"]
    ratio = gpt2.get_vocab_size() / tokenizer.get_vocab_size()
    print("Bir xil matn uchun bizniki GPT-2 ga nisbatan {:+.1%} token ishlatadi".format(rel))
    print("Lug'atimiz esa {:.0f} barobar kichik.".format(ratio))
    print()
    if rel <= 0:
        print("Bizniki YAXSHIROQ siqmoqda - lug'ati kichik bo'lsa ham. Sababi:")
        print("GPT-2 butun internetga o'rgatilgan, biz esa aynan shu korpusga.")
        print("Tor domen -> har bir slot foydali -> samarali lug'at.")
    else:
        print("Bizniki biroz ko'proq token ishlatmoqda, lekin lug'ati ancha kichik.")
        print("Bu kutilgan savdo: vocab_size ni oshirsangiz farq kamayadi.")
        print("VOCAB_SIZE ni {} dan oshirib ko'ring - vazifaning 1-topshirig'i.".format(
            tokenizer.get_vocab_size()))
except Exception as e:
    print("GPT-2 tokenizerini yuklab bo'lmadi (internet yo'qmi?):", e)

Tokenizer                    vocab     fertility   compression
--------------------------------------------------------------
bizniki                      3,106         1.412         4.448
GPT-2                       50,257         1.350         4.652

Bir xil matn uchun bizniki GPT-2 ga nisbatan +4.6% token ishlatadi
Lug'atimiz esa 16 barobar kichik.

Bizniki biroz ko'proq token ishlatmoqda, lekin lug'ati ancha kichik.
Bu kutilgan savdo: vocab_size ni oshirsangiz farq kamayadi.
VOCAB_SIZE ni 3106 dan oshirib ko'ring - vazifaning 1-topshirig'i.


In [18]:
demo = "Once upon a time there was a curious littl tokenizer."

enc = tokenizer.encode(demo)
print("BIZNIKI  ({} token):".format(len(enc.ids)))
print(" ", enc.tokens)
print("  IDs:", enc.ids)
print()

try:
    g = gpt2.encode(demo)
    print("GPT-2    ({} token):".format(len(g.ids)))
    print(" ", g.tokens)
except NameError:
    pass

BIZNIKI  (23 token):
  ['O', 'nce', 'Ġup', 'on', 'Ġa', 'Ġtime', 'Ġthere', 'Ġw', 'as', 'Ġa', 'Ġc', 'uri', 'ous', 'Ġl', 'it', 't', 'l', 'Ġto', 'k', 'en', 'iz', 'er', '.']
  IDs: [47, 2482, 743, 264, 257, 414, 951, 276, 362, 257, 274, 1602, 843, 315, 283, 84, 76, 286, 75, 279, 904, 275, 14]

GPT-2    (14 token):
  ['Once', 'Ġupon', 'Ġa', 'Ġtime', 'Ġthere', 'Ġwas', 'Ġa', 'Ġcurious', 'Ġl', 'itt', 'l', 'Ġtoken', 'izer', '.']


In [19]:
out_of_domain = [
    "Once upon a time a little rabbit found a red ball.",   # o'z domenimiz
    "def encode(self, text: str) -> List[int]: return self._bpe(text)",
    "Kecha men O'zbekistonda bo'ldim va juda ko'p narsa o'rgandim.",
    "The mitochondrion is the powerhouse of the eukaryotic cell.",
]

head = "{:>12}{:>12}  {}".format("tokenlar", "belgi/token", "matn")
print(head)
print("-" * 70)
for t in out_of_domain:
    n = len(tokenizer.encode(t).ids)
    print("{:>12}{:>12.2f}  {}".format(n, len(t) / n, t[:44]))

print()
print("Bolalar hikoyasi eng samarali siqiladi - lug'at aynan shunga o'rgatilgan.")
print("Kod va o'zbek tili yomon siqiladi: model ularni ko'rmagan.")
print("Xulosa: tokenizer - korpusning ko'zgusi, universal narsa emas.")

    tokenlar belgi/token  matn
----------------------------------------------------------------------
          22        2.27  Once upon a time a little rabbit found a red
          36        1.78  def encode(self, text: str) -> List[int]: re
          39        1.56  Kecha men O'zbekistonda bo'ldim va juda ko'p
          25        2.36  The mitochondrion is the powerhouse of the e

Bolalar hikoyasi eng samarali siqiladi - lug'at aynan shunga o'rgatilgan.
Kod va o'zbek tili yomon siqiladi: model ularni ko'rmagan.
Xulosa: tokenizer - korpusning ko'zgusi, universal narsa emas.


# DATASET BUILDER

## 9. Split BEFORE encoding

In [21]:
indices = list(range(len(docs)))
rng = random.Random(SEED)
rng.shuffle(indices)

n_val = max(1, int(len(docs) * VAL_RATIO))
val_idx   = indices[:n_val]
train_idx = indices[n_val:]

train_docs = [docs[i] for i in train_idx]
val_docs   = [docs[i] for i in val_idx]

print("Train hujjatlar : {:,}".format(len(train_docs)))
print("Val hujjatlar   : {:,}  ({:.2%})".format(len(val_docs), len(val_docs) / len(docs)))
print()

assert set(train_idx).isdisjoint(set(val_idx)), "Split kesishib ketdi!"
assert len(train_idx) + len(val_idx) == len(docs), "Hujjatlar yo'qoldi!"
print("Split toza: kesishuv yo'q, hujjat yo'qolmadi.")

Train hujjatlar : 169
Val hujjatlar   : 1  (0.59%)

Split toza: kesishuv yo'q, hujjat yo'qolmadi.
